**Title**: Run local analysis and upload back to Flywheel  
**Date**:  15-Apr-2020  
**Description**:  
Find and download input file from flywheel, process locally, create an analysis container and upload local process outputs to it.

Topics that will be covered:
* Find file in Flywheel Acquisition container
* Download file and run analysis locally
* Create Analysis container
* Local outputs are uploaded to the Flywheel Analysis container.
    

# Requirements
- Access to Flywheel Project that contains Acquisitions to analyze locally
- Have at least Read-Write Permission on Project level

# Setup

In [ ]:
# Install specific packages required for this notebook
!pip install flywheel-sdk nipype

In [ ]:
# Import packages
from getpass import getpass
import logging
import os
from pathlib import Path

import flywheel
import nipype
from nipype.interfaces.image import Reorient

In [ ]:
# Instantiate a logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('root')

# Flywheel API Key and Client

Get your API_KEY. More on this at in the Flywheel SDK doc [here](https://flywheel-io.gitlab.io/product/backend/sdk/branches/master/python/getting_started.html#api-key).

In [ ]:
API_KEY = getpass('Enter API_KEY here: ')

Instantiate the Flywheel API client

In [ ]:
fw = flywheel.Client(API_KEY if 'API_KEY' in locals() else os.environ.get('FW_KEY'))

Show Flywheel logging information

In [ ]:
# Initialize user information 
USER_INFO = fw.get_current_user()

In [ ]:
log.info('You are now logged in as %s to %s', USER_INFO['email'], fw.get_config()['site']['api_url'])

# Constants

In [ ]:
# Flywheel path to the acquistion container with the file we want to process locally
FW_PATH_TO_ACQ = '<you-group>/<your-project>/<subject.label>/<session.label>/<acquisition.label>'
# Filename of the file we want to process locally
FILENAME = '<the-input-filename-here.ext>'
# Path where the input files will be download
DOWNLOAD_PATH = '/tmp'

# Check User Permission

Here we will be checking if you have the right permission on your Flywheel Instance to proceed in this notebook.

For this notebook, we will be checking your Group and Project Permission

Define `check_container_permission` to ensure the user has the right permission

In [ ]:
def check_container_permission(container_type, container_id, user_id):
    if container_type == 'group':
        permission = fw.get_group_user_permission(container_id, user_id)
    elif container_type == 'project':
        permission = fw.get_project_user_permission(container_id, user_id)
    
    if permission['access'] == 'rw' or permission['access'] == 'admin':
        return log.info('You can proceed to the next step.')
    else:
        return log.warning('Please contact your site admin to get read/write permission in order to proceed.')

In [ ]:
GROUP_LABEL = input('Enter your Group Label here:')

In [ ]:
PROJECT_LABEL = input('Enter your Project Label here: ')

In [ ]:
if 'user' or 'developer' in user_info['roles']:
    group = fw.lookup(f'{GROUP_LABEL}')
    project = fw.projects.find_first(f'label={PROJECT_LABEL}')
    if group:
        check_permission('group', group.id, user_info['_id'])
        if project:
            check_permission('project', project.id, user_info['_id'])
        else:
            raise ValueError(f'Project {PROJECT_LABEL} is not valid. Please pick another project label.')
    else:
        raise ValueError(f'Group {GROUP_LABEL} is not valid. Please pick another group label.')
    
else:
    log.info('You can proceed to the next step.')

# Main script

## Find file

Find the Flywheel Acquisition container by performing a lookup 

In [ ]:
acquisition = fw.lookup(FW_PATH_TO_ACQ)

Find the File in that Acquisition container by name

In [ ]:
file = acquisition.get_file(FILENAME)

## Download file locally

In [ ]:
dest_path = str(Path(DOWNLOAD_PATH) / FILENAME)

In [ ]:
file.download(dest_path)

## Process file locally

This is a very simple processing which is just reorienting the nifti image

In [ ]:
reorient = Reorient(orientation='LPS')
reorient.inputs.in_file = dest_path
res = reorient.run()
out_file = res.outputs.out_file
log.info('Output file saved to: %s', out_file)

## Create Analysis container

Create an Analysis container attached to the Session with reference to the input files

In [ ]:
session = fw.get_session(acquisition.parents.session)
analysis = session.add_analysis(label='My Analysis label', inputs=[file.ref()])

## Upload the output file to Analysis container

In [ ]:
analysis.upload_output(out_file)

## Check the uploaded file

In [ ]:
analysis = analysis.reload()

In [ ]:
assert analysis.files[0].name == os.path.basename(out_file)
assert analysis.files[0].size > 0